# YOLOv8 Training v Google Colab (z již anotovaných dat)

Tento notebook je training-only verze.

Použití:
1. Nahrajte složku `annotations_combined` na Google Drive do `MyDrive/MLproject/`
2. Spusťte buňky postupně
3. Výstupem bude natrénovaný model `best.pt` + metriky

## 1) Připojení Google Drive a nastavení cest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/MLproject')
DATA_DIR = PROJECT_DIR / 'annotations_combined'
IMAGES_DIR = DATA_DIR / 'images'
LABELS_DIR = DATA_DIR / 'labels'
WORK_DIR = PROJECT_DIR / 'colab_training'
SPLIT_DIR = WORK_DIR / 'dataset_split'

for d in [WORK_DIR, SPLIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR exists:', DATA_DIR.exists())
print('IMAGES_DIR exists:', IMAGES_DIR.exists())
print('LABELS_DIR exists:', LABELS_DIR.exists())

## 2) Instalace a import knihoven pro trénování

In [ ]:
!pip -q install ultralytics pyyaml

In [ ]:
import random
import shutil
from pathlib import Path
import yaml
from ultralytics import YOLO

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('Importy OK. Ultralytics ready.')

## 3) Kontrola dat a příprava seznamu párů image-label

In [ ]:
image_exts = {'.jpg', '.jpeg', '.png'}
all_images = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in image_exts]

pairs = []
missing_labels = []
for img in all_images:
    label = LABELS_DIR / f'{img.stem}.txt'
    if label.exists():
        pairs.append((img, label))
    else:
        missing_labels.append(img.name)

print(f'Obrazky celkem: {len(all_images)}')
print(f'Validni pary image+label: {len(pairs)}')
print(f'Chybejici labely: {len(missing_labels)}')
if missing_labels[:10]:
    print('Ukazka chybejicich labelu:', missing_labels[:10])

## 4) Split dat (train/val/test = 70/20/10)

In [ ]:
split_root = SPLIT_DIR
for subset in ['train', 'val', 'test']:
    (split_root / 'images' / subset).mkdir(parents=True, exist_ok=True)
    (split_root / 'labels' / subset).mkdir(parents=True, exist_ok=True)

random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.7)
n_val = int(n * 0.2)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train+n_val]
test_pairs = pairs[n_train+n_val:]

for subset_name, subset_pairs in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]:
    for img, lbl in subset_pairs:
        shutil.copy2(img, split_root / 'images' / subset_name / img.name)
        shutil.copy2(lbl, split_root / 'labels' / subset_name / lbl.name)

print('Train:', len(train_pairs))
print('Val:', len(val_pairs))
print('Test:', len(test_pairs))

## 5) Vytvoření dataset.yaml

In [ ]:
dataset_yaml_path = split_root / 'dataset.yaml'
dataset_cfg = {
    'path': str(split_root),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': ['hasici', 'policie', 'zachranka']
}

with open(dataset_yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(dataset_cfg, f, sort_keys=False, allow_unicode=True)

print('dataset.yaml vytvoren:', dataset_yaml_path)
print(dataset_yaml_path.read_text(encoding='utf-8'))

## 6) Trénování YOLOv8 modelu

In [ ]:
EPOCHS = 50
IMGSZ = 640
BATCH = 16
MODEL_NAME = 'yolov8m.pt'

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(dataset_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(WORK_DIR / 'runs'),
    name='izs_detector',
    pretrained=True,
    patience=20,
    verbose=True
)

print('Trenovani dokonceno.')

## 7) Vyhodnocení modelu (mAP)

In [ ]:
best_model_path = WORK_DIR / 'runs' / 'izs_detector' / 'weights' / 'best.pt'
print('best_model_path exists:', best_model_path.exists())

best_model = YOLO(str(best_model_path))
val_metrics = best_model.val(data=str(dataset_yaml_path), split='test')

print('mAP50:', float(val_metrics.box.map50))
print('mAP50-95:', float(val_metrics.box.map))

## 8) Rychlý test predikce na pár obrázcích

In [ ]:
sample_test_images = list((split_root / 'images' / 'test').glob('*'))[:5]
print('Počet test obrázků pro ukázku:', len(sample_test_images))

pred_out = best_model.predict(
    source=[str(p) for p in sample_test_images],
    conf=0.25,
    save=True,
    project=str(WORK_DIR / 'predictions'),
    name='sample_preds'
)

print('Predikce hotova. Vystupy jsou v:', WORK_DIR / 'predictions' / 'sample_preds')

## 9) Export modelu

In [ ]:
EXPORT_DIR = WORK_DIR / 'export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

best_copy = EXPORT_DIR / 'best.pt'
shutil.copy2(best_model_path, best_copy)

best_model.export(format='onnx')

print('Export hotovy:')
print('- best.pt:', best_copy)
print('- runs:', WORK_DIR / 'runs')
print('- predictions:', WORK_DIR / 'predictions')